In [1]:
import pandas as pd
from fastai.collab import *
from fastai.tabular.all import *
import numpy as np

In [2]:
path = untar_data(URLs.ML_100k)

In [3]:
path

Path('C:/Users/enesm/.fastai/data/ml-100k')

In [4]:
ratings = pd.read_csv(path/'u.data', delimiter='\t', header=None,
                      names=['user','movie','rating','timestamp'])
ratings.head()

,user,movie,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [5]:
movies = pd.read_csv(path/'u.item',  delimiter='|', encoding='latin-1',
                     usecols=(0,1), names=('movie','title'), header=None)
movies.head()

,movie,title
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)


In [6]:
ratings = ratings.merge(movies)
ratings.head()

,user,movie,rating,timestamp,title
0,196,242,3,881250949,Kolya (1996)
1,63,242,3,875747190,Kolya (1996)
2,226,242,5,883888671,Kolya (1996)
3,154,242,3,879138235,Kolya (1996)
4,306,242,5,876503793,Kolya (1996)


In [7]:
dls= CollabDataLoaders.from_df(ratings,item_name='title',bs=64)
dls.show_batch()

,user,title,rating
0,535,Secrets & Lies (1996),4
1,348,"Beautician and the Beast, The (1997)",3
2,769,2 Days in the Valley (1996),3
3,325,Indiana Jones and the Last Crusade (1989),2
4,788,Phenomenon (1996),3
5,447,Screamers (1995),4
6,279,Cinema Paradiso (1988),3
7,427,Air Force One (1997),4
8,496,Mr. Smith Goes to Washington (1939),1
9,13,Muppet Treasure Island (1996),3


In [8]:
n_users=len(dls.classes['user'])
n_movies=len(dls.classes['title'])
n_factors=5

user_factors=torch.randn(n_users,n_factors)
movie_factors=torch.randn(n_movies,n_factors)

In [9]:
movie_factors.shape, user_factors.shape

(torch.Size([1665, 5]), torch.Size([944, 5]))

In [10]:
one_hot_3 = one_hot(3, n_users).float()
one_hot_3

tensor([0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 

In [11]:
user_factors

tensor([[ 0.1198, -0.4265, -1.0377,  0.2220,  0.3763],
        [ 0.9037, -0.1361,  2.2357,  0.8403,  0.2009],
        [-0.6662, -2.0643, -0.0053, -0.1025,  0.2618],
        ...,
        [-0.4258, -2.1464,  0.0196,  1.3111,  0.1634],
        [-1.3895,  0.6869, -1.0430,  1.0689,  0.3643],
        [ 0.8246,  0.1572,  0.1628,  1.3259,  0.8129]])

In [12]:
user_factors.t()

tensor([[ 0.1198,  0.9037, -0.6662,  ..., -0.4258, -1.3895,  0.8246],
        [-0.4265, -0.1361, -2.0643,  ..., -2.1464,  0.6869,  0.1572],
        [-1.0377,  2.2357, -0.0053,  ...,  0.0196, -1.0430,  0.1628],
        [ 0.2220,  0.8403, -0.1025,  ...,  1.3111,  1.0689,  1.3259],
        [ 0.3763,  0.2009,  0.2618,  ...,  0.1634,  0.3643,  0.8129]])

In [13]:
args=[[1,2,3,4,5,6,7],[1,2,3,4,5,6,7]]

In [14]:
user_factors.t() @ one_hot_3

tensor([ 0.9234, -1.4737, -1.2285, -0.6599,  0.3673])

In [15]:
class DotProduct(Module):
    def __init__(self, n_users, n_movies, n_factors):
        self.user_factors = Embedding(n_users, n_factors)
        self.movie_factors = Embedding(n_movies, n_factors)
        
    def forward(self, x):
        users = self.user_factors(x[:,0])
        movies = self.movie_factors(x[:,1])
        return (users * movies).sum(dim=1)

In [31]:
x,y = dls.one_batch()
x.shape

torch.Size([64, 2])

In [35]:
model = DotProduct(n_users, n_movies, 50)
learn = Learner(dls, model, loss_func=MSELossFlat())

In [38]:
learn.fit_one_cycle(5,5e-3)

epoch,train_loss,valid_loss,time
0,1.337568,1.292572,00:04
1,1.081021,1.102064,00:04
2,0.977352,0.993810,00:04
3,0.847234,0.896146,00:04
4,0.788397,0.878367,00:04


In [20]:
#Creating our own embedding Module


class T(Module):
    def __init__(self):self.a= torch.ones(3)

L(T().parameters())

(#0) []

In [21]:
class T(Module):
    def __init__(self): self.a = nn.Parameter(torch.ones(3))

L(T().parameters())

(#1) [Parameter containing:
tensor([1., 1., 1.], requires_grad=True)]

In [22]:
class T(Module):
    def __init__(self): self.a= nn.Linear(1,3,bias=False)
t=T()
L(t.parameters())

(#1) [Parameter containing:
tensor([[ 0.1787],
        [-0.8832],
        [-0.6437]], requires_grad=True)]

In [23]:
T().a

Linear(in_features=1, out_features=3, bias=False)

In [24]:
type(t.a.weight)

torch.nn.parameter.Parameter

In [25]:
def create_params(size):
    return nn.Parameter(torch.zeros(*size).normal_(0,0.01))

In [29]:
class DotProductBias(Module):
    def __init__(self,n_users,n_movies,n_factors, y_range=(0,5.5)):
        self.user_factors=create_params([n_users,n_factors])
        self.user_bias=create_params([n_users])
        self.movie_factors=create_params([n_movies,n_factors])
        self.movie_bias=create_params([n_movies])
        self.y_range=y_range

    def forward(self,x):
        users=self.user_factors[x[:,0]]
        movies=self.movie_factors[x[:,1]]
        res=(users*movies).sum(dim=1)
        res+=self.user_bias[x[:,0]]+self.movie_bias[x[:,1]]
        return sigmoid_range(res, *self.y_range)

In [30]:
model= DotProductBias(n_users,n_movies,50)
learn=Learner(dls,model,loss_func=MSELossFlat())
learn.fit_one_cycle(5,5e-3,wd=0.1)

epoch,train_loss,valid_loss,time
0,0.914762,0.937730,00:07
1,0.845642,0.866867,00:05
2,0.734300,0.830808,00:04
3,0.562838,0.815738,00:04
4,0.461842,0.816748,00:05


In [ ]:
import torch

user_factors = torch.tensor([[0.2, 0.4],
                             [0.5, 0.1],
                             [0.9, 0.3]])

movie_factors = torch.tensor([[0.7, 0.6,2],
                              [0.1, 0.8,3],
                              [0.3, 0.2,4],
                              [0.9, 0.7,5]])


In [ ]:
x = torch.tensor([[1, 2],   # User index: 1, Movie index: 2
                  [0, 3],   # User index: 0, Movie index: 3
                  [2, 1]])  # User index: 2, Movie index: 1

In [ ]:
users = [[0.5, 0.1],  # User index 1
         [0.2, 0.4],  # User index 0
         [0.9, 0.3]]  # User index 2

In [ ]:
movie_factors[x[:,1]]

tensor([[0.3000, 0.2000, 4.0000],
        [0.9000, 0.7000, 5.0000],
        [0.1000, 0.8000, 3.0000]])

In [ ]:
users

tensor([[0.5000, 0.1000],
        [0.2000, 0.4000],
        [0.9000, 0.3000]])

In [ ]:
class CollabNN(Module):
    def __init__(self, user_sz, item_sz, y_range=(0,5.5), n_act=100):
        self.user_factors = Embedding(*user_sz)
        self.item_factors = Embedding(*item_sz)
        self.layers = nn.Sequential(
            nn.Linear(user_sz[1]+item_sz[1], n_act),
            nn.ReLU(),
            nn.Linear(n_act, 1))
        self.y_range = y_range
        
    def forward(self, x):
        embs = self.user_factors(x[:,0]),self.item_factors(x[:,1])
        x = self.layers(torch.cat(embs, dim=1))
        return sigmoid_range(x, *self.y_range)